# 01: Import

## 01-1: 라이브러리 & 데이터

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import ast

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [54]:
data_path_raw = '/content/drive/MyDrive/epoch_datathon/data/raw'
data_path_write = '/content/drive/MyDrive/epoch_datathon/data'

In [56]:
df_credits_raw = pd.read_csv(f'{data_path_raw}/credits.csv')
df_movies_metadata_raw = pd.read_csv(f'{data_path_raw}/movies_metadata.csv')
df_ratings_raw = pd.read_csv(f'{data_path_raw}/ratings.csv')

# merged_df_cast_direct_popul_raw = pd.read_csv(f'{data_path_write}/merged_df_cast_direct_popul.csv')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

## 01-2: 전처리 함수 정의

In [57]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 결측치 보완 함수
def handle_missing_values(df, column, method='mean', fill_value=None):
    if method == 'mean':
        df[column].fillna(df[column].mean(), inplace=True)
    elif method == 'median':
        df[column].fillna(df[column].median(), inplace=True)
    elif method == 'mode':
        df[column].fillna(df[column].mode()[0], inplace=True)
    elif method == 'constant':
        df[column].fillna(fill_value, inplace=True)
    return df

# 이상치 처리 함수
def handle_outliers(df, column, lower_quantile=0.05, upper_quantile=0.95):
    lower_bound = df[column].quantile(lower_quantile)
    upper_bound = df[column].quantile(upper_quantile)
    df[column] = df[column].clip(lower_bound, upper_bound)
    return df

# 열 정규화 함수 (Min-Max scaling)
def normalize_column(df, column):
    scaler = MinMaxScaler()
    df[[column]] = scaler.fit_transform(df[[column]])
    return df

# 열 표준화 함수 (Z-score scaling)
def standardize_column(df, column):
    scaler = StandardScaler()
    df[[column]] = scaler.fit_transform(df[[column]])
    return df

# 훈련 세트와 테스트 세트 나누는 함수
def split_train_test(df, target_column, test_size=0.2, random_state=42):
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

# 사용 예시:
"""
df = handle_missing_values(df, 'age', method='median')
df = handle_outliers(df, 'income', lower_quantile=0.01, upper_quantile=0.99)
df = normalize_column(df, 'price')
df = standardize_column(df, 'height')
X_train, X_test, y_train, y_test = split_train_test(df, 'target', test_size=0.25)
"""

"\ndf = handle_missing_values(df, 'age', method='median')\ndf = handle_outliers(df, 'income', lower_quantile=0.01, upper_quantile=0.99)\ndf = normalize_column(df, 'price')\ndf = standardize_column(df, 'height')\nX_train, X_test, y_train, y_test = split_train_test(df, 'target', test_size=0.25)\n"

In [58]:
# 특정 행 삭제 함수
def drop_rows_by_condition(df, column, condition):
    """
    주어진 조건에 따라 특정 열에서 행을 삭제하는 함수.

    Parameters:
    - df (pd.DataFrame): 처리할 데이터프레임.
    - column (str): 조건을 확인할 열의 이름.
    - condition (callable): 열의 값에 대한 조건 함수 (True일 때 해당 행이 삭제됨).

    Returns:
    - pd.DataFrame: 조건에 맞는 행이 삭제된 새로운 데이터프레임.
    """
    # 조건에 맞는 행 삭제
    filtered_df = df[~df[column].apply(condition)]
    return filtered_df

# 사용 예시: 나이가 30 이상인 사람들의 행을 삭제하는 함수 호출
"""
new_df = drop_rows_by_condition(df, 'Age', lambda x: x >= 30)
print(new_df)
"""

"\nnew_df = drop_rows_by_condition(df, 'Age', lambda x: x >= 30)\nprint(new_df)\n"

In [59]:
# 특정 열 삭제 함수
def drop_columns_by_name(df, columns):
    """
    주어진 열 이름에 따라 열을 삭제하는 함수.

    Parameters:
    - df (pd.DataFrame): 처리할 데이터프레임.
    - columns (list of str): 삭제할 열의 이름 리스트.

    Returns:
    - pd.DataFrame: 지정된 열이 삭제된 새로운 데이터프레임.
    """
    # 열 삭제
    filtered_df = df.drop(columns=columns)
    return filtered_df

# 사용 예시: 'Score' 열을 삭제하는 함수 호출
"""
new_df = drop_columns_by_name(df, ['Score'])
print(new_df)
"""

"\nnew_df = drop_columns_by_name(df, ['Score'])\nprint(new_df)\n"

# 02: 1차 전처리

In [60]:
# 데이터 복제
df_credits = df_credits_raw.copy()
df_movies_metadata = df_movies_metadata_raw.copy()
df_ratings = df_ratings_raw.copy()

In [63]:
# df_movies_metadata_BeforeDateCut: 날짜 범위 지정 전
df_movies_metadata_BeforeDateCut = df_movies_metadata.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

## 02-1: 정원님 전처리

### 02-1-1: Status

In [64]:
# status 열에서 값이 Released 가 아닌 값들 확인
other_status = df_movies_metadata_BeforeDateCut[~df_movies_metadata_BeforeDateCut['status'].isin(['Released'])]

# Released 아닌 값들 출력
print(other_status[['status']])

                status
189                NaN
682            Rumored
767                NaN
775            Rumored
1032           Rumored
...                ...
45109          Rumored
45159              NaN
45207  Post Production
45289          Rumored
45442              NaN

[452 rows x 1 columns]


In [67]:
# status 열에서 Released 가 아닌 값들 삭제
df_movies_metadata_BeforeDateCut_Released = df_movies_metadata_BeforeDateCut[(df_movies_metadata_BeforeDateCut['status'] == 'Released')]

# Released 만 남긴 데이터 확인
df_movies_metadata_BeforeDateCut_Released.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45014 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45014 non-null  object 
 1   belongs_to_collection  4466 non-null   object 
 2   budget                 45014 non-null  object 
 3   genres                 45014 non-null  object 
 4   homepage               7706 non-null   object 
 5   id                     45014 non-null  object 
 6   imdb_id                44999 non-null  object 
 7   original_language      45004 non-null  object 
 8   original_title         45014 non-null  object 
 9   overview               44094 non-null  object 
 10  popularity             45014 non-null  object 
 11  poster_path            44641 non-null  object 
 12  production_companies   45014 non-null  object 
 13  production_countries   45014 non-null  object 
 14  release_date           44936 non-null  object 
 15  revenue

### 02-1-2: Release date

#### Avatar `release_date` 수정
- 틀린 값

In [68]:
# 수정 전
df_movies_metadata_BeforeDateCut[df_movies_metadata_BeforeDateCut['title'] == 'Avatar']

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
14551,False,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.avatarmovie.com/,19995,tt0499549,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",...,2009-12-10,2.787965e+09,162.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Enter the World of Pandora.,Avatar,False,7.2,12114.0


In [69]:
# Avatar 개봉일 수정
df_movies_metadata_BeforeDateCut.loc[df_movies_metadata_BeforeDateCut['id'] == '19995', 'release_date'] = '2009-12-18'

# 수정 후
df_movies_metadata_BeforeDateCut[df_movies_metadata_BeforeDateCut['title'] == 'Avatar']

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
14551,False,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.avatarmovie.com/,19995,tt0499549,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",...,2009-12-18,2.787965e+09,162.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Enter the World of Pandora.,Avatar,False,7.2,12114.0


#### `release_date` dtype 수정

In [70]:
# release_date를 datetime 형식으로 변환 (Null 값은 NaT 으로 변환)
df_movies_metadata_BeforeDateCut_Released['release_date'] = pd.to_datetime(
    df_movies_metadata_BeforeDateCut_Released['release_date'], errors='coerce')

<ipython-input-70-33ecad58b002>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies_metadata_BeforeDateCut_Released['release_date'] = pd.to_datetime(


#### Avatar 이후 결측치는 직접 채워 넣음

In [71]:
# Avatar 이후 release_date 결측치 확인
df_movies_metadata[df_movies_metadata['release_date'].isna()][['release_date']]

len(df_movies_metadata[df_movies_metadata['release_date'].isna()][['release_date']]) # 87개

87

In [73]:
# 영화 이름 리스트와 개별적으로 조사한 release dates 를 사용하여 결측치 보완

# 영화 목록과 각 영화의 release date를 담은 dictionary
matching_movies = ["Human Failure", "Dreamkiller", "Endeavour", "Dance of Outlaws", "Bad Chicken",
                        "Getting Back to Abnormal", "Disaster Playground", "Yedyanchi Jatra", "Monk by Blood",
                        "I Am Syd Stone", "Vous êtes très jolie, mademoiselle", "Vogelfrei",
                        "And Then There Were None", "Pawn's Move",
                        "If These Knishes Could Talk: The Story of the New York Accent", "Friends and Romans",
                        "Digital Dharma", "The Last Gold", "Scott Hall: Living on a Razor's Edge",
                        "Bad Dad Rehab", "Lo Sound Desert", "Pad Yatra: A Green Odyssey", "Always Faithful",
                        "Allende en su laberinto", "Dolpo Tulku - Heimkehr in den Himalaya", "Winning Favour",
                        "When the Day Had No Name", "Jedi Junior High", "Irwin & Fran", "Shivering Trunks",
                        "Neither Wolf Nor Dog", "Mundo Cão", "The Garden of Afflictions",
                        "All Superheroes Must Die 2: The Last Superhero", "Subdue"]

# 개별적으로 조사한 release dates
matching_release_dates = ["2008-04-08", "2010-11-16", "2013-01-02", "2012-08-08", "2013-04-09",
                          "2013-03-09", "2015-03-17", "2012-03-16", "2013-09-10", "2020-10-01",
                          "2014-03-04", "2007-10-04", "2015-12-26", "2011-03-07", "2013-02-21",
                          "2014-11-21", "2012-06-16", "2016-07-11", "2016-07-05", "2016-07-03",
                          "2015-03-14", "2012-03-08", "2014-11-11", "2014-11-06", "2010-09-09",
                          "2012-04-10", "2017-03-29", "2014-03-07", "2013-11-04", "2012-01-01",
                          "2016-04-28", "2016-03-17", "2017-05-18", "2016-01-01", "2016-10-10"]
len(matching_release_dates)

# 데이터프레임에서 영화 이름에 해당하는 release_date를 업데이트 하는 함수
def fill_missing_release_dates(df, movie_list, release_dates):
    for movie, release_date in zip(movie_list, release_dates):
        # 영화 제목이 있는 행의 release_date 값을 업데이트
        df.loc[df['title'] == movie, 'release_date'] = release_date
    return df

# 영화 데이터프레임에서 결측치 보완
df_movies_metadata = fill_missing_release_dates(df_movies_metadata, matching_movies, matching_release_dates)

# 결측치가 잘 채워졌는지 확인
df_movies_metadata[df_movies_metadata['title'] == "Human Failure"]
df_movies_metadata[df_movies_metadata['title'] == "Subdue"]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
45461,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,2016-10-10,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0


#### 블록버스터 영화의 release_date를 기준으로 행 제거
- Avatar 개봉일(2009-12-18) 이전 행 제거

In [75]:
# 선정한 7개 블록버스터 영화의 id
bb_list = ['19995', '10193', '24428', '68721', '109445', '135397', '140607']

# 블록버스터 영화 데이터 추출
blockbuster = df_movies_metadata_BeforeDateCut[df_movies_metadata_BeforeDateCut['id'].isin(bb_list)]

# 블록버스터 영화 확인
blockbuster

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
14551,False,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.avatarmovie.com/,19995,tt0499549,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",...,2009-12-18,2.787965e+09,162.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Enter the World of Pandora.,Avatar,False,7.2,12114.0
15348,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",200000000,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",http://disney.go.com/toystory/,10193,tt0435761,en,Toy Story 3,"Woody, Buzz, and the rest of Andy's toys haven...",...,2010-06-16,1.066970e+09,103.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,No toy gets left behind.,Toy Story 3,False,7.6,4710.0
17818,False,"{'id': 86311, 'name': 'The Avengers Collection...",220000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",http://marvel.com/avengers_movie/,24428,tt0848228,en,The Avengers,When an unexpected enemy emerges and threatens...,...,2012-04-25,1.519558e+09,143.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Some assembly required.,The Avengers,False,7.4,12000.0
20830,False,"{'id': 131292, 'name': 'Iron Man Collection', ...",200000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://marvel.com/ironman3,68721,tt1300854,en,Iron Man 3,When Tony Stark's world is torn apart by a for...,...,2013-04-18,1.215440e+09,130.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Unleash the power behind the armor.,Iron Man 3,False,6.8,8951.0
22110,False,"{'id': 386382, 'name': 'Frozen Collection', 'p...",150000000,"[{'id': 16, 'name': 'Animation'}, {'id': 12, '...",http://movies.disney.com/frozen,109445,tt2294629,en,Frozen,Young princess Anna of Arendelle dreams about ...,...,2013-11-27,1.274219e+09,102.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Only the act of true love will thaw a frozen h...,Frozen,False,7.3,5440.0
25084,False,"{'id': 328, 'name': 'Jurassic Park Collection'...",150000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.jurassicworld.com/,135397,tt0369610,en,Jurassic World,Twenty-two years after the events of Jurassic ...,...,2015-06-09,1.513529e+09,124.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The park is open.,Jurassic World,False,6.5,8842.0
26555,False,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,en,Star Wars: The Force Awakens,Thirty years after defeating the Galactic Empi...,...,2015-12-15,2.068224e+09,136.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.5,7993.0


In [79]:
# release_date 값이 2009-12-18(Avatar 개봉일) 이전인 데이터 필터링

# 2009-12-18 이전의 release_date 값 필터링
df_movies_metadata = df_movies_metadata_BeforeDateCut_Released[
    (df_movies_metadata_BeforeDateCut_Released['release_date'] >= '2009-12-18') &
    (df_movies_metadata_BeforeDateCut_Released['release_date'].notna())
]

# 결과 확인
df_movies_metadata.info()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
868,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,79782,tt1684935,en,Wenecja,An atmospheric coming-of-age story featuring a...,...,2010-05-25,0.0,110.0,"[{'iso_639_1': 'pl', 'name': 'Polski'}]",Released,NaN,Venice,False,7.5,4.0
1081,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...",NaN,141210,tt2250194,en,The Sleepover,"The town of Derry has a secret, but no one tol...",...,2013-10-12,0.0,6.0,[],Released,NaN,The Sleepover,False,8.0,1.0
2114,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",http://www.thefarmerswifefilm.co.uk/,143750,tt2140519,en,The Farmer's Wife,"As her surroundings are invaded by outsiders, ...",...,2012-06-20,0.0,18.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,The Farmer's Wife,False,10.0,1.0
2564,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,2012-03-22,0.0,84.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,One Nation. Underfed.,A Place at the Table,False,6.9,7.0
2778,False,NaN,0,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",NaN,171982,tt2378428,en,Romance,She's as hot as Britney Spears. Hotter. She pa...,...,2012-10-09,0.0,27.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Romance,False,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45438,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,327237,tt3814486,nl,"Bloed, Zweet en Tranen","Bloed, Zweet en Tranen (Blood, Sweat and Tears...",...,2015-04-02,0.0,111.0,"[{'iso_639_1': 'nl', 'name': 'Nederlands'}]",Released,The movie about Andre Hazes,"Blood, Sweat and Tears",False,6.8,11.0
45453,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,404604,tt5690142,hi,Maa,The bliss of a biology teacher’s family life i...,...,2017-07-07,0.0,146.0,"[{'iso_639_1': 'hi', 'name': 'हिन्दी'}]",Released,NaN,Mom,False,6.6,14.0
45454,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,420346,tt4130180,en,The Morning After,The Morning After is a feature film that consi...,...,2015-01-11,0.0,79.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,What happened last night?,The Morning After,False,4.0,2.0
45462,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0


### 02-1-3: `Runtime`

In [77]:
# runtime 결측치(Null) 확인
invalid_runtimes = df_movies_metadata[df_movies_metadata['runtime'].isnull()]

invalid_runtimes[['runtime']].isnull().sum() # 90개

,0
runtime,90


In [80]:
# runtime 결측치(0) 확인
zero_runtimes = df_movies_metadata[df_movies_metadata['runtime'] == 0]

(zero_runtimes[['runtime']]==0).sum() # 524개

,0
runtime,524


In [82]:
# runtime이 0인 값들을 NaN으로 변경
df_movies_metadata['runtime'] = df_movies_metadata['runtime'].replace(0, np.nan)

# 변경된 결과 확인
zero_runtimes = df_movies_metadata[df_movies_metadata['runtime'] == 0]
print((zero_runtimes[['runtime']] == 0).sum())  # 0개 출력 (모두 NaN으로 변경됨)

runtime    0
dtype: int64


In [81]:
# 미니시리즈/다부작 필터링

# runtime 값이 255 이상인 데이터 필터링 (날짜 범위 지정 이후 runtime 255 이상은 모두 미니시리즈/다부작)
df_movies_metadata_DropMiniseries1 = drop_rows_by_condition(df_movies_metadata, 'runtime', lambda x: x >= 255)

# runtime 값이 254 이하인 데이터 중 미니시리즈/다부작 영화 13개 필터링

# 제외할 title 리스트
miniseries_titles = ['World Without End', 'Wolf Hall', 'Tut', 'The Untold History of the United States',
                  'The Story of Film: An Odyssey', 'The Sixtie', 'The Saboteurs', 'The Roosevelts: An Intimate History',
                  'The Pacific', 'The Men Who Built America', 'The Keepers', 'The Bible', 'Texas Rising']

# 13개 데이터 필터링
df_movies_metadata_DropMiniseries2 = drop_rows_by_condition(df_movies_metadata_DropMiniseries1, 'title', lambda x: x in miniseries_titles)

df_movies_metadata = df_movies_metadata_DropMiniseries1

# 결과 출력
df_movies_metadata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12626 entries, 868 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  12626 non-null  object        
 1   belongs_to_collection  1027 non-null   object        
 2   budget                 12626 non-null  object        
 3   genres                 12626 non-null  object        
 4   homepage               4654 non-null   object        
 5   id                     12626 non-null  object        
 6   imdb_id                12623 non-null  object        
 7   original_language      12623 non-null  object        
 8   original_title         12626 non-null  object        
 9   overview               12341 non-null  object        
 10  popularity             12626 non-null  object        
 11  poster_path            12553 non-null  object        
 12  production_companies   12626 non-null  object        
 13  prod

### 02-1-4: `Title`

#### title 비교하여 중복 행 제거

In [83]:
# title 결측치 확인
df_movies_metadata['title'].isnull().sum() # 없음

0

In [84]:
# title 중복된 값들 확인 (title 별로 그룹화 및 카운트)
duplicate_movies = df_movies_metadata['title'].value_counts().sort_values(ascending=False)

In [85]:
# title 중복된 행들의 release_date 값을 확인하기 위한 필터링 (우선 영화명과 개봉일이 같은 것들부터 확인)
duplicate_movies_release = df_movies_metadata[df_movies_metadata['title'].isin(duplicate_movies.index.tolist())]

# 영화 제목별로 그룹화하여 release_date 중복 확인
duplicates_with_same_release = duplicate_movies_release.groupby(['title', 'release_date']).size().reset_index(name='count')

# release_date 값이 2개 이상인 경우를 필터링
duplicates_with_same_release = duplicates_with_same_release[duplicates_with_same_release['count'] > 1]

# 결과 확인
print(duplicates_with_same_release)

                      title release_date  count
434    A Place at the Table   2012-03-22      2
1661             Black Gold   2011-12-21      2
2053   Camille Claudel 1915   2013-03-13      2
2170  Cemetery of Splendour   2015-09-02      2
3725          Force Majeure   2014-08-15      2
9753           The Congress   2013-05-16      2


In [86]:
# title & release_date 중복 데이터 전체 컬럼 확인
df_movies_metadata[df_movies_metadata['title'].isin(duplicates_with_same_release['title'])].sort_values(by='title')

# 모든 컬럼이 중복

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
2564,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,2012-03-22,0.0,84.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,One Nation. Underfed.,A Place at the Table,False,6.9,7.0
21116,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,2012-03-22,0.0,84.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,One Nation. Underfed.,A Place at the Table,False,6.9,7.0
11155,False,NaN,40000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",NaN,77221,tt1701210,en,Black Gold,"On the Arabian Peninsula in the 1930s, two war...",...,2011-12-21,5446000.0,130.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Black Gold,False,5.9,77.0
20843,False,NaN,40000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",NaN,77221,tt1701210,en,Black Gold,"On the Arabian Peninsula in the 1930s, two war...",...,2011-12-21,5446000.0,130.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Black Gold,False,5.9,77.0
4356,False,NaN,3512454,"[{'id': 18, 'name': 'Drama'}]",NaN,110428,tt2018086,fr,Camille Claudel 1915,"Winter, 1915. Confined by her family to an asy...",...,2013-03-13,115860.0,95.0,"[{'iso_639_1': 'fr', 'name': 'Français'}]",Released,NaN,Camille Claudel 1915,False,7.0,20.0
23534,False,NaN,3512454,"[{'id': 18, 'name': 'Drama'}]",NaN,110428,tt2018086,fr,Camille Claudel 1915,"Winter, 1915. Confined by her family to an asy...",...,2013-03-13,115860.0,95.0,"[{'iso_639_1': 'fr', 'name': 'Français'}]",Released,NaN,Camille Claudel 1915,False,7.0,20.0
33184,False,NaN,980000,"[{'id': 18, 'name': 'Drama'}, {'id': 14, 'name...",NaN,298721,tt2818654,th,รักที่ขอนแก่น,"In a hospital, ten soldiers are being treated ...",...,2015-09-02,0.0,122.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,NaN,Cemetery of Splendour,False,4.4,50.0
40040,False,NaN,980000,"[{'id': 18, 'name': 'Drama'}, {'id': 14, 'name...",NaN,298721,tt2818654,th,รักที่ขอนแก่น,"In a hospital, ten soldiers are being treated ...",...,2015-09-02,0.0,122.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,NaN,Cemetery of Splendour,False,4.4,50.0
24164,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,265189,tt2121382,sv,Turist,"While holidaying in the French Alps, a Swedish...",...,2014-08-15,1359497.0,118.0,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,NaN,Force Majeure,False,6.8,255.0
45265,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,265189,tt2121382,sv,Turist,"While holidaying in the French Alps, a Swedish...",...,2014-08-15,1359497.0,118.0,"[{'iso_639_1': 'fr', 'name': 'Français'}, {'is...",Released,NaN,Force Majeure,False,6.8,255.0


In [87]:
# 중복인 행 중 하나를 삭제
df_movies_metadata = df_movies_metadata.drop_duplicates(subset=['title', 'release_date'], keep='first')

# 결과 확인
df_movies_metadata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12620 entries, 868 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   adult                  12620 non-null  object        
 1   belongs_to_collection  1027 non-null   object        
 2   budget                 12620 non-null  object        
 3   genres                 12620 non-null  object        
 4   homepage               4654 non-null   object        
 5   id                     12620 non-null  object        
 6   imdb_id                12617 non-null  object        
 7   original_language      12617 non-null  object        
 8   original_title         12620 non-null  object        
 9   overview               12335 non-null  object        
 10  popularity             12620 non-null  object        
 11  poster_path            12547 non-null  object        
 12  production_companies   12620 non-null  object        
 13  prod

### 02-1-5: `Ratings`

In [88]:
# df_ratings_raw 에서 영화별 rating 의 평균 & 개수 새로 그룹화 -> movies_ratings_Groupby
movies_ratings_Groupby = df_ratings_raw.groupby('movieId')['rating'].agg(['count', 'mean']).reset_index()

# movies_ratings_Groupby 컬럼명 rename
movies_ratings_Groupby = movies_ratings_Groupby.rename(columns={'count': 'rating_count',
                                                                'mean': 'rating_average'})

# 결과 확인
print(movies_ratings_Groupby.head())

   movieId  rating_count  rating_average
0        1         66008        3.888157
1        2         26060        3.236953
2        3         15497        3.175550
3        4          2981        2.875713
4        5         15258        3.079565


In [92]:
data_path_write = '/content/drive/MyDrive/epoch_datathon/data'


df_movies_metadata.to_csv(f'{data_path_write}/fin_df_cleaned.csv', index=False)
movies_ratings_Groupby.to_csv(f'{data_path_write}/fin_movies_ratings_Groupby.csv', index=False)

---

**협: 여기서부터는 아직 실행하지 말아주세요 (코드가 오래 걸립니다)**

---

# 03: API를 통한 결측치 보완
- API Key: 26d745f34e7e5e7859c3ea736e802874

## 03-0: Import Data
- 02까지 전처리된 `fin_df_cleaned.csv` 파일로 시작

In [102]:
df_meta_cleaned_raw = pd.read_csv(f'{data_path_write}/fin_df_cleaned.csv')
df_ratings_cleaned_raw = pd.read_csv(f'{data_path_write}/fin_movies_ratings_Groupby.csv.csv')

df_meta_cleaned_raw.info()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/epoch_datathon/data/fin_movies_ratings_Groupby.csv.csv'

In [98]:
df_meta_cleaned = df_meta_cleaned_raw.copy()
df_ratings_cleaned = df_ratings_cleaned_raw.copy()

## 03-1: Revenue

### 03-1-1: Revenue API Imputation

In [99]:
import requests

api_key = '26d745f34e7e5e7859c3ea736e802874'

In [100]:
import requests
from tqdm import tqdm  # 진행률 표시를 위한 라이브러리

# TMDB API를 통해 영화 revenue 값을 가져오는 함수 정의
def fetch_revenue_from_api(movie_id):
    if pd.isna(movie_id):
        return None
    try:
        # TMDB 영화 API URL 구성
        url = f"https://api.themoviedb.org/3/movie/{int(movie_id)}"
        params = {
            "api_key": api_key,
            "language": "en-US"
        }
        response = requests.get(url, params=params)
        data = response.json()

        # revenue 값을 반환 (없을 경우 None 반환)
        return data.get('revenue', None)

    except (requests.exceptions.RequestException, ValueError):
        return None

In [101]:
# tqdm을 사용하여 진행률을 표시하면서 'id' 컬럼을 통해 revenue 값을 가져옴
tqdm.pandas()  # tqdm이 pandas의 progress_apply와 연동되도록 설정

df_meta_cleaned['revenue_api'] = df_meta_cleaned['id'].progress_apply(fetch_revenue_from_api)

# 결과 확인
print(df_meta_cleaned[['id', 'revenue', 'revenue_api']].head())

100%|██████████| 12620/12620 [30:30<00:00,  6.89it/s]

       id  revenue  revenue_api
0   79782      0.0          0.0
1  141210      0.0          0.0
2  143750      0.0          0.0
3   84198      0.0     230522.0
4  171982      0.0          0.0


In [107]:
# save point

data_path_write = '/content/drive/MyDrive/epoch_datathon/data'

df_meta_cleaned.to_csv(f'{data_path_write}/fin_df_cleaned_v1.csv', index=False)

### 03-1-2: Revenue_api Imputation 점검

In [104]:
df_meta_cleaned[['id', 'revenue', 'revenue_api']]

,id,revenue,revenue_api
0,79782,0.0,0.0
1,141210,0.0,0.0
2,143750,0.0,0.0
3,84198,0.0,230522.0
4,171982,0.0,0.0
...,...,...,...
12615,455661,0.0,0.0
12616,327237,0.0,0.0
12617,404604,0.0,0.0
12618,420346,0.0,0.0


In [105]:
filtered_df = df_meta_cleaned[
    (df_meta_cleaned['revenue'] == 0) & (df_meta_cleaned['revenue_api'] != 0)
]

filtered_df[['id', 'budget','revenue', 'revenue_api']]  # Display the desired columns

,id,budget,revenue,revenue_api
3,84198,0,0.0,230522.0
7,78022,0,0.0,5850000.0
20,33511,0,0.0,6577779.0
30,7978,150000000,0.0,140700000.0
36,41894,0,0.0,82739.0
...,...,...,...,...
12583,395767,0,0.0,NaN
12585,248705,25868826,0.0,18552314.0
12588,188421,0,0.0,NaN
12592,426272,0,0.0,2583.0


In [106]:
df_meta_cleaned[(df_meta_cleaned['revenue'] == 0)]

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,revenue_api
0,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,79782,tt1684935,en,Wenecja,An atmospheric coming-of-age story featuring a...,...,0.0,110.0,"[{'iso_639_1': 'pl', 'name': 'Polski'}]",Released,NaN,Venice,False,7.5,4.0,0.0
1,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...",NaN,141210,tt2250194,en,The Sleepover,"The town of Derry has a secret, but no one tol...",...,0.0,6.0,[],Released,NaN,The Sleepover,False,8.0,1.0,0.0
2,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",http://www.thefarmerswifefilm.co.uk/,143750,tt2140519,en,The Farmer's Wife,"As her surroundings are invaded by outsiders, ...",...,0.0,18.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,The Farmer's Wife,False,10.0,1.0,0.0
3,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,0.0,84.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,One Nation. Underfed.,A Place at the Table,False,6.9,7.0,230522.0
4,False,NaN,0,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...",NaN,171982,tt2378428,en,Romance,She's as hot as Britney Spears. Hotter. She pa...,...,0.0,27.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Romance,False,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12615,False,NaN,0,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...",NaN,455661,tt6969946,en,In a Heartbeat,A closeted boy runs the risk of being outed by...,...,0.0,4.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Heart Wants What The Heart Wants,In a Heartbeat,False,8.3,146.0,0.0
12616,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,327237,tt3814486,nl,"Bloed, Zweet en Tranen","Bloed, Zweet en Tranen (Blood, Sweat and Tears...",...,0.0,111.0,"[{'iso_639_1': 'nl', 'name': 'Nederlands'}]",Released,The movie about Andre Hazes,"Blood, Sweat and Tears",False,6.8,11.0,0.0
12617,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,404604,tt5690142,hi,Maa,The bliss of a biology teacher’s family life i...,...,0.0,146.0,"[{'iso_639_1': 'hi', 'name': 'हिन्दी'}]",Released,NaN,Mom,False,6.6,14.0,0.0
12618,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,420346,tt4130180,en,The Morning After,The Morning After is a feature film that consi...,...,0.0,79.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,What happened last night?,The Morning After,False,4.0,2.0,0.0


## 03-2: Budget

### 03-2-1: Budget API Imputation

In [108]:
df_meta_cleaned_v1_raw = pd.read_csv(f'{data_path_write}/fin_df_cleaned_v1.csv')

In [109]:
df_meta_cleaned_v1 = df_meta_cleaned_v1_raw.copy()
df_meta_cleaned_v1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12620 entries, 0 to 12619
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  12620 non-null  bool   
 1   belongs_to_collection  1027 non-null   object 
 2   budget                 12620 non-null  int64  
 3   genres                 12620 non-null  object 
 4   homepage               4654 non-null   object 
 5   id                     12620 non-null  int64  
 6   imdb_id                12617 non-null  object 
 7   original_language      12617 non-null  object 
 8   original_title         12620 non-null  object 
 9   overview               12335 non-null  object 
 10  popularity             12620 non-null  float64
 11  poster_path            12547 non-null  object 
 12  production_companies   12620 non-null  object 
 13  production_countries   12620 non-null  object 
 14  release_date           12620 non-null  object 
 15  re

In [111]:
import asyncio
import aiohttp
import pandas as pd
from tqdm import tqdm
import nest_asyncio

# 이미 실행 중인 이벤트 루프에 비동기 처리를 허용
nest_asyncio.apply()

# 비동기 TMDB API에서 budget을 가져오는 함수
async def fetch_budget_from_api(session, movie_id, api_key):
    if pd.isna(movie_id):
        return None
    try:
        # TMDB 영화 API URL 구성
        url = f"https://api.themoviedb.org/3/movie/{int(movie_id)}"
        params = {
            "api_key": api_key,
            "language": "en-US"
        }
        async with session.get(url, params=params) as response:
            if response.status == 200:
                data = await response.json()
                # budget 값을 반환 (없을 경우 None 반환)
                return data.get('budget', None)
            else:
                return None
    except (aiohttp.ClientError, ValueError):
        return None

# 비동기 요청을 실행할 메인 함수
async def fetch_all_budgets(df, api_key):
    async with aiohttp.ClientSession() as session:
        # 비동기적으로 API 요청을 보내기 위한 태스크 리스트 생성
        tasks = []
        for movie_id in tqdm(df['id'], desc="Fetching budgets"):
            tasks.append(fetch_budget_from_api(session, movie_id, api_key))

        # 모든 태스크를 비동기적으로 처리
        return await asyncio.gather(*tasks)

# 비동기 함수를 호출하여 budget_api 컬럼 생성
def get_budgets_async(df, api_key):
    # 이미 실행 중인 이벤트 루프에서 비동기 작업을 실행
    loop = asyncio.get_event_loop()
    budgets = loop.run_until_complete(fetch_all_budgets(df, api_key))
    df['budget_api'] = budgets
    return df

# 실행 예시
df_meta_cleaned_v1 = get_budgets_async(df_meta_cleaned_v1, api_key)

# 결과 확인
print(df_meta_cleaned_v1[['id', 'budget', 'budget_api']].head())

Fetching budgets: 100%|██████████| 12620/12620 [00:00<00:00, 512163.68it/s]


       id  budget  budget_api
0   79782       0   1783810.0
1  141210       0         0.0
2  143750       0         0.0
3   84198       0         0.0
4  171982       0         0.0


### 03-2-2: Buget_api Imputaion 점검

In [112]:
filtered_df = df_meta_cleaned_v1[
    (df_meta_cleaned_v1['budget'] == 0) & (df_meta_cleaned_v1['budget_api'] != 0)
]

filtered_df[['id', 'budget','budget_api', 'revenue', 'revenue_api']]  # Display the desired columns

,id,budget,budget_api,revenue,revenue_api
0,79782,0,1783810.0,0.0,0.0
10,16166,0,1114000.0,0.0,0.0
26,13477,0,55000000.0,36699403.0,43042835.0
36,41894,0,10000000.0,0.0,82739.0
37,22166,0,5000000.0,0.0,0.0
...,...,...,...,...,...
12531,254689,0,NaN,0.0,NaN
12545,451955,0,7110000.0,0.0,4248574.0
12547,410554,0,10000000.0,0.0,0.0
12583,395767,0,NaN,0.0,NaN


In [115]:
# save point
data_path_write = '/content/drive/MyDrive/epoch_datathon/data'

df_meta_cleaned_v1.to_csv(f'{data_path_write}/fin_df_cleaned_v2.csv', index=False)

## 03-3: `Revenue_api`, `Budget_api`에 대한 검증
> 기존 자료인 `budget`, `revenue` 각각과 api를 통해 가져온 열인 `revenue_api`, `budget_api`의 값이 합리적인지

In [ ]:
# 여기는 정원님 파트

## 03-4: 주연급 배우, 감독의 popularity API

### 03-4-1: 배우

#### 03-4-1-1: 주연급 배우 ID 추출
> JSON 형식인 cast열에서 order가 0,1,2인 값에 대한 popluarity를 API로 추출해본 결과, `order==0`인 경우는 결측이 많아서 `order==1`인 배우, `order==2`인 배우의 popularity 활용


In [116]:
df_meta_cleaned_v2 = pd.read_csv(f'{data_path_write}/fin_df_cleaned_v2.csv')
# df_credits_raw = pd.read_csv(f'{data_path_write}/fin_credits.csv')
df_meta_cleaned_v2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12620 entries, 0 to 12619
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  12620 non-null  bool   
 1   belongs_to_collection  1027 non-null   object 
 2   budget                 12620 non-null  int64  
 3   genres                 12620 non-null  object 
 4   homepage               4654 non-null   object 
 5   id                     12620 non-null  int64  
 6   imdb_id                12617 non-null  object 
 7   original_language      12617 non-null  object 
 8   original_title         12620 non-null  object 
 9   overview               12335 non-null  object 
 10  popularity             12620 non-null  float64
 11  poster_path            12547 non-null  object 
 12  production_companies   12620 non-null  object 
 13  production_countries   12620 non-null  object 
 14  release_date           12620 non-null  object 
 15  re

In [117]:
df_credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [118]:
# cast 열에서 order가 0, 1, 2인 배우의 ID를 추출하는 함수
def extract_cast_members(cast_data):
    try:
        # cast 열을 문자열로부터 리스트로 변환
        cast_list = ast.literal_eval(cast_data)

        # order가 0인 배우의 ID를 cast_1에, 1인 배우의 ID를 cast_2에 할당
        cast_0 = next((str(int(member['id'])) for member in cast_list if member.get('order') == 0), None)
        cast_1 = next((str(int(member['id'])) for member in cast_list if member.get('order') == 1), None)
        cast_2 = next((str(int(member['id'])) for member in cast_list if member.get('order') == 2), None)

        return pd.Series([cast_0, cast_1, cast_2])
    except (ValueError, SyntaxError, TypeError):
        # JSON 변환 실패 시 None 반환
        return pd.Series([None, None, None])

In [119]:
# `cast` 열에 함수를 적용하여 `cast_1`과 `cast_2` 열 생성
df_credits[['cast_0', 'cast_1', 'cast_2']] = df_credits['cast'].apply(extract_cast_members)

# 결과 확인
print(df_credits[['id', 'cast_0', 'cast_1', 'cast_2']].head())

/usr/lib/python3.10/ast.py:50: RuntimeWarning: coroutine 'fetch_all_budgets' was never awaited
  return compile(source, filename, mode, flags,


      id cast_0 cast_1 cast_2
0    862     31  12898   7167
1   8844   2157   8537    205
2  15602   6837   3151  13567
3  31357   8851   9780  18284
4  11862  67773   3092    519


In [120]:
# 머지
df_meta_cleaned_v2['id'] = df_meta_cleaned_v2['id'].astype(str)
df_credits['id'] = df_credits['id'].astype(str)

merged_df = pd.merge(df_meta_cleaned_v2, df_credits, on='id', how='left')
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12631 entries, 0 to 12630
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  12631 non-null  bool   
 1   belongs_to_collection  1027 non-null   object 
 2   budget                 12631 non-null  int64  
 3   genres                 12631 non-null  object 
 4   homepage               4656 non-null   object 
 5   id                     12631 non-null  object 
 6   imdb_id                12628 non-null  object 
 7   original_language      12628 non-null  object 
 8   original_title         12631 non-null  object 
 9   overview               12346 non-null  object 
 10  popularity             12631 non-null  float64
 11  poster_path            12558 non-null  object 
 12  production_companies   12631 non-null  object 
 13  production_countries   12631 non-null  object 
 14  release_date           12631 non-null  object 
 15  re

#### 03-4-1-2: popularity API 호출

In [121]:
import asyncio
import aiohttp
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
import nest_asyncio

# nest_asyncio로 이미 실행 중인 이벤트 루프를 다시 사용할 수 있게 함
nest_asyncio.apply()

# 비동기 I/O로 API에서 배우의 인기 점수를 가져오는 함수
async def fetch_popularity_score_async(session, person_id):
    if person_id is None:
        return None
    url = f"https://api.themoviedb.org/3/person/{person_id}"
    params = {
        "api_key": api_key,  # 실제 API 키를 설정해야 합니다.
        "language": "en-US"
    }
    try:
        async with session.get(url, params=params) as response:
            data = await response.json()
            return data.get("popularity")
    except Exception as e:
        print(f"Error fetching data for person_id {person_id}: {e}")
        return None

# 전체 배우 리스트에 대해 비동기적으로 API 호출을 하는 함수
async def fetch_all_popularity_scores(person_ids):
    async with aiohttp.ClientSession() as session:
        tasks = []
        # 프로그레스바와 함께 비동기 작업을 관리
        for person_id in person_ids:
            task = fetch_popularity_score_async(session, person_id)
            tasks.append(task)

        # tqdm을 사용하여 진행률 표시
        return await tqdm_asyncio.gather(*tasks, desc="Fetching popularity scores")

# 비동기 처리를 동기적으로 호출하는 함수
def fetch_popularity_scores(person_ids):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(fetch_all_popularity_scores(person_ids))

In [ ]:
# # 첫 5개의 행만 추출하여 처리하는 예시 코드
# subset_df = merged_df.head(5)  # merged_df의 첫 5개 행을 가져옵니다.

# # cast_1에 대한 popularity 점수를 비동기적으로 가져와 cast_1_popularity 열에 추가
# subset_df['cast_1_popularity'] = fetch_popularity_scores(subset_df['cast_1'])

# # 결과 확인
# print(subset_df[['cast_1', 'cast_1_popularity']])

In [122]:
# cast_1과 cast_2에 대해 각각의 popularity 점수를 비동기적으로 가져와 새로운 열로 추가
merged_df['cast_1_popularity'] = fetch_popularity_scores(merged_df['cast_1'])
merged_df['cast_2_popularity'] = fetch_popularity_scores(merged_df['cast_2'])

# 결과 확인
print(merged_df[['id', 'cast_1', 'cast_1_popularity', 'cast_2', 'cast_2_popularity']].head())

Fetching popularity scores: 100%|██████████| 12631/12631 [02:28<00:00, 84.89it/s]

       id   cast_1  cast_1_popularity   cast_2  cast_2_popularity
0   79782   591258              2.955   140221              1.330
1  141210  1114468              3.401  1114469              0.109
2  143750    11855             29.404   976738             13.209
3   84198   219917              3.013  1155683              0.001
4   84198   219917              3.013  1155683              0.001


In [123]:
# save point

data_path_write = '/content/drive/MyDrive/epoch_datathon/data'

merged_df.to_csv(f'{data_path_write}/fin_df_cleaned_v3.csv', index=False)

### 03-4-2: 감독

In [124]:
df_meta_cleaned_v3 = pd.read_csv(f'{data_path_write}/fin_df_cleaned_v3.csv')

df_meta_cleaned_v3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12631 entries, 0 to 12630
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  12631 non-null  bool   
 1   belongs_to_collection  1027 non-null   object 
 2   budget                 12631 non-null  int64  
 3   genres                 12631 non-null  object 
 4   homepage               4656 non-null   object 
 5   id                     12631 non-null  int64  
 6   imdb_id                12628 non-null  object 
 7   original_language      12628 non-null  object 
 8   original_title         12631 non-null  object 
 9   overview               12346 non-null  object 
 10  popularity             12631 non-null  float64
 11  poster_path            12558 non-null  object 
 12  production_companies   12631 non-null  object 
 13  production_countries   12631 non-null  object 
 14  release_date           12631 non-null  object 
 15  re

#### 03-4-2-1: 감독 ID 추출

In [125]:
# 감독의 ID를 추출하는 함수 정의
def extract_director_id(crew_data):
    try:
        # crew 열을 문자열에서 리스트로 변환
        crew_list = ast.literal_eval(crew_data)

        # job이 "Director"인 인물의 ID 추출
        director_id = next((member['id'] for member in crew_list if member.get('job') == "Director"), None)

        return director_id
    except (ValueError, SyntaxError):
        return None

In [126]:
# crew 열에 함수를 적용하여 director_id 열 생성
df_meta_cleaned_v3['director_id'] = df_meta_cleaned_v3['crew'].apply(extract_director_id)

# 결과 확인
print(df_meta_cleaned_v3[['id', 'director_id']].head())

       id  director_id
0   79782     587971.0
1  141210    1114467.0
2  143750     192852.0
3   84198     638550.0
4   84198     638550.0


#### 03-4-2-2: 감독 popularity API 호출

In [127]:
import asyncio
import aiohttp
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
import nest_asyncio

# nest_asyncio로 이미 실행 중인 이벤트 루프를 다시 사용할 수 있게 함
nest_asyncio.apply()

# 비동기 I/O로 API에서 감독의 인기 점수를 가져오는 함수
async def fetch_director_popularity_score_async(session, person_id):
    if person_id is None:
        return None
    url = f"https://api.themoviedb.org/3/person/{person_id}"
    params = {
        "api_key": api_key,  # 실제 API 키를 설정해야 합니다.
        "language": "en-US"
    }
    try:
        async with session.get(url, params=params) as response:
            data = await response.json()
            return data.get("popularity")
    except Exception as e:
        print(f"Error fetching data for person_id {person_id}: {e}")
        return None

# 전체 감독 리스트에 대해 비동기적으로 API 호출을 하는 함수
async def fetch_all_director_popularity_scores(person_ids):
    async with aiohttp.ClientSession() as session:
        tasks = []
        # 프로그레스바와 함께 비동기 작업을 관리
        for person_id in person_ids:
            task = fetch_director_popularity_score_async(session, person_id)
            tasks.append(task)

        # tqdm을 사용하여 진행률 표시
        return await tqdm_asyncio.gather(*tasks, desc="Fetching director popularity scores")

# 비동기 처리를 동기적으로 호출하는 함수
def fetch_director_popularity_scores(person_ids):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(fetch_all_director_popularity_scores(person_ids))

# # 예시 데이터프레임 (실제 데이터는 credits_df 사용)
# credits_df = pd.DataFrame({
#     'id': [1, 2, 3],
#     'director_id': [12898, 8537, None]  # 감독의 ID 값들
# })

In [128]:
# tqdm의 pandas 확장을 사용
tqdm_asyncio.pandas()

# 감독의 인기 점수를 비동기적으로 가져와서 데이터프레임에 추가하는 함수
def add_director_popularity_scores(df):
    person_ids = df['director_id'].tolist()
    df['popularity_score'] = fetch_director_popularity_scores(person_ids)
    return df

# 감독의 인기 점수를 데이터프레임에 추가
df_meta_cleaned_v3 = add_director_popularity_scores(df_meta_cleaned_v3)
print(df_meta_cleaned_v3)

Fetching director popularity scores: 100%|██████████| 12631/12631 [03:26<00:00, 61.07it/s]

       adult belongs_to_collection  budget  \
0      False                   NaN       0   
1      False                   NaN       0   
2      False                   NaN       0   
3      False                   NaN       0   
4      False                   NaN       0   
...      ...                   ...     ...   
12626  False                   NaN       0   
12627  False                   NaN       0   
12628  False                   NaN       0   
12629  False                   NaN       0   
12630  False                   NaN       0   

                                                  genres  \
0      [{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...   
1      [{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...   
2                          [{'id': 18, 'name': 'Drama'}]   
3                    [{'id': 99, 'name': 'Documentary'}]   
4                    [{'id': 99, 'name': 'Documentary'}]   
...                                                  ...   
12626  [{'id': 10751, 'name

In [133]:
df_meta_cleaned_v3.rename(columns={'popularity_score': 'director_popularity'}, inplace=True)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,budget_api,cast,crew,cast_0,cast_1,cast_2,cast_1_popularity,cast_2_popularity,director_id,director_popularity
0,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,79782,tt1684935,en,Wenecja,An atmospheric coming-of-age story featuring a...,...,1783810.0,"[{'cast_id': 1005, 'character': 'Marek', 'cred...","[{'credit_id': '52fe49e5c3a368484e145fb7', 'de...",587975.0,591258.0,140221.0,2.955,1.330,587971.0,0.330
1,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...",NaN,141210,tt2250194,en,The Sleepover,"The town of Derry has a secret, but no one tol...",...,0.0,"[{'cast_id': 2, 'character': 'Rachel', 'credit...","[{'credit_id': '52fe4aaf9251416c750ea6f1', 'de...",1067761.0,1114468.0,1114469.0,3.401,0.109,1114467.0,0.451
2,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",http://www.thefarmerswifefilm.co.uk/,143750,tt2140519,en,The Farmer's Wife,"As her surroundings are invaded by outsiders, ...",...,0.0,"[{'cast_id': 10, 'character': 'The Auctioneer'...","[{'credit_id': '52fe4b169251416c750f7cd5', 'de...",1120400.0,11855.0,976738.0,29.404,13.209,192852.0,2.015
3,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,0.0,"[{'cast_id': 3, 'character': 'Himself', 'credi...","[{'credit_id': '52fe48e09251416c9109b347', 'de...",1229.0,219917.0,1155683.0,3.013,0.001,638550.0,0.556
4,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,84198,tt1736049,en,A Place at the Table,"Using personal stories, this powerful document...",...,0.0,"[{'cast_id': 3, 'character': 'Himself', 'credi...","[{'credit_id': '52fe48e09251416c9109b347', 'de...",1229.0,219917.0,1155683.0,3.013,0.001,638550.0,0.556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12626,False,NaN,0,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...",NaN,455661,tt6969946,en,In a Heartbeat,A closeted boy runs the risk of being outed by...,...,0.0,[],"[{'credit_id': '5981a15c92514151e0011b51', 'de...",NaN,NaN,NaN,NaN,NaN,1809041.0,NaN
12627,False,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,327237,tt3814486,nl,"Bloed, Zweet en Tranen","Bloed, Zweet en Tranen (Blood, Sweat and Tears...",...,0.0,"[{'cast_id': 2, 'character': 'André Hazes', 'c...","[{'credit_id': '54edb0e0c3a3686d58003a13', 'de...",NaN,1388985.0,128859.0,0.560,1.818,1109734.0,0.168
12628,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,404604,tt5690142,hi,Maa,The bliss of a biology teacher’s family life i...,...,0.0,"[{'cast_id': 1, 'character': 'Devki Sabarwal',...","[{'credit_id': '58ee55bbc3a3683df500bd0f', 'de...",109549.0,1254942.0,87328.0,13.541,12.343,1644440.0,NaN
12629,False,NaN,0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,420346,tt4130180,en,The Morning After,The Morning After is a feature film that consi...,...,0.0,"[{'cast_id': 0, 'character': 'Lauren', 'credit...","[{'credit_id': '587626f4c3a3682b33008299', 'de...",NaN,1736940.0,1302424.0,0.854,1.190,1736944.0,0.046


In [7]:
# save point

data_path_write = '/content/drive/MyDrive/epoch_datathon/data'

df_meta_cleaned_v3.to_csv(f'{data_path_write}/fin_df_cleaned_v4.csv', index=False)

# 04: 최종 전처리

In [8]:
df_meta_cleaned_v4 = pd.read_csv(f'{data_path_write}/fin_df_cleaned_v4.csv')

df_meta_cleaned_v4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12631 entries, 0 to 12630
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  12631 non-null  bool   
 1   belongs_to_collection  1027 non-null   object 
 2   budget                 12631 non-null  int64  
 3   genres                 12631 non-null  object 
 4   homepage               4656 non-null   object 
 5   id                     12631 non-null  int64  
 6   imdb_id                12628 non-null  object 
 7   original_language      12628 non-null  object 
 8   original_title         12631 non-null  object 
 9   overview               12346 non-null  object 
 10  popularity             12631 non-null  float64
 11  poster_path            12558 non-null  object 
 12  production_companies   12631 non-null  object 
 13  production_countries   12631 non-null  object 
 14  release_date           12631 non-null  object 
 15  re

## 04-2: Budget 결측치 보완
> API 활용 후에도 여전히 결측인 (revenue 값이 0이 아닌 행 중에서) `budget` 값에 대한 처리

### 04-2-1: `budget` 결측치를 `vote_count`를 통해 imputation

In [9]:
# budget 과 다른 컬럼들과의 각각 상관관계 확인

corr_column_list = ['popularity','revenue','runtime','vote_average','vote_count','cast_1_popularity',
                    'cast_2_popularity','director_popularity','revenue_api','budget_api']

# 각 상관계수를 리스트에 저장
correlations_with_budget = []

for column in corr_column_list:
    correlation = df_meta_cleaned_v4['budget'].corr(df_meta_cleaned_v4[column])
    correlations_with_budget.append((column, correlation))

# DataFrame으로 변환
correlations_with_budget = pd.DataFrame(correlations_with_budget, columns=['Column', 'Correlation_with_budget'])

# 결과 확인
print(correlations_with_budget) # vote_count와의 상관관계가 유의미하게 높음 (0.754701)

                Column  Correlation_with_budget
0           popularity                 0.419618
1              revenue                 0.827523
2              runtime                 0.199370
3         vote_average                 0.077011
4           vote_count                 0.754701
5    cast_1_popularity                 0.341018
6    cast_2_popularity                 0.367735
7  director_popularity                 0.286641
8          revenue_api                 0.829572
9           budget_api                 0.990096


In [10]:
# 상관계수 유의미하게 높은 vote_count 로 훈련하기 전 결측치 확인

# vote_count Null 인 값들 확인
invalid_vote_counts = df_meta_cleaned_v4[df_meta_cleaned_v4['vote_count'].isnull()]

print(invalid_vote_counts[['vote_count']])

# vote_count 0 인 값들 확인
zero_vote_counts = df_meta_cleaned_v4[df_meta_cleaned_v4['vote_count'] == '0']

print(zero_vote_counts[['vote_count']])

Empty DataFrame
Columns: [vote_count]
Index: []
Empty DataFrame
Columns: [vote_count]
Index: []


In [11]:
# vote_count 와 budget 의 상관계수를 통해 budget 0 값 새로운 값으로 채우는 코드

from sklearn.linear_model import LinearRegression

# budget이 0이 아닌 행들 필터링
non_zero_budgets = df_meta_cleaned_v4[df_meta_cleaned_v4['budget'] != 0]

# 선형회귀를 위한 데이터 준비
X_train = non_zero_budgets[['vote_count']]
y_train = non_zero_budgets['budget']

# 선형회귀모델 훈련
budget_votecount_lin_model = LinearRegression()
budget_votecount_lin_model.fit(X_train, y_train)

# budget이 0인 행 확인
zero_budget_rows = df_meta_cleaned_v4[df_meta_cleaned_v4['budget'] == 0]

if not zero_budget_rows.empty:
    # budget 값이 0 이라면 새로운 값 예측
    X_test = zero_budget_rows[['vote_count']]
    predicted_budgets = budget_votecount_lin_model.predict(X_test)

    # 새로운 값 채워 넣기
    df_meta_cleaned_v4.loc[df_meta_cleaned_v4['budget'] == 0, 'budget'] = predicted_budgets

# 결과 확인
print(df_meta_cleaned_v4[['vote_count', 'budget']].head())

   vote_count        budget
0         4.0  9.808471e+06
1         1.0  9.739400e+06
2         1.0  9.739400e+06
3         7.0  9.877543e+06
4         7.0  9.877543e+06


<ipython-input-11-6aef90b589b6>:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 9808471.43669724  9739399.51439513  9739399.51439513 ...
 10038711.17770426  9762423.48849583  9716375.54029442]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_meta_cleaned_v4.loc[df_meta_cleaned_v4['budget'] == 0, 'budget'] = predicted_budgets


In [12]:
# budget 원래 0이 아닌 값 예시 확인 (훈련 후) -> 훈련 후 budget이 0인 값만 채워졌는지 확인
df_meta_cleaned_v4[df_meta_cleaned_v4['budget'] == 0] # budget이 0인 값만 채워짐

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,budget_api,cast,crew,cast_0,cast_1,cast_2,cast_1_popularity,cast_2_popularity,director_id,director_popularity


## 04-3: 사용하지 않을 컬럼 제거
> 최종 활용 컬럼:
### metadata
`budget`
`id`
`revenue`
`runtime`
`vote_average`
`vote_count`
`bb_effect_14`
`bb_diff_14`
`bb_effect_28`
`bb_diff_28`

### credit
`cast_1_popularity`
`cast_2_popularity`
`director_popularity`

### ratings
`rating_count`
`rating_average`

In [13]:
df_meta_cleaned_v4.columns

# 삭제할 열
# ['adult', 'belongs_to_collection',  'genres', 'homepage',
#  'imdb_id', 'original_language', 'original_title', 'overview','poster_path', 'production_companies',
#  'production_countries', 'spoken_languages', 'status', 'tagline', 'video',
#  'cast', 'crew', 'cast_0', 'cast_1', 'cast_2', 'director_id']

# 유지할 열

selected_columns = ['budget', 'id', 'popularity', 'release_date', 'revenue', 'runtime',
                   'title', 'vote_average', 'vote_count', 'revenue_api', 'budget_api',
                   'cast_1_popularity', 'cast_2_popularity', 'director_popularity']

# Create a new DataFrame with only the selected columns
df_meta_cleaned_v5 = df_meta_cleaned_v4[selected_columns]

In [14]:
df_meta_cleaned_v5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12631 entries, 0 to 12630
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   budget               12631 non-null  float64
 1   id                   12631 non-null  int64  
 2   popularity           12631 non-null  float64
 3   release_date         12631 non-null  object 
 4   revenue              12631 non-null  float64
 5   runtime              12017 non-null  float64
 6   title                12631 non-null  object 
 7   vote_average         12631 non-null  float64
 8   vote_count           12631 non-null  float64
 9   revenue_api          12502 non-null  float64
 10  budget_api           12497 non-null  float64
 11  cast_1_popularity    10890 non-null  float64
 12  cast_2_popularity    10551 non-null  float64
 13  director_popularity  11856 non-null  float64
dtypes: float64(11), int64(1), object(2)
memory usage: 1.3+ MB


# 05: 블록버스터 영화 개봉일 파생 변수 생성
**Case 1: 14일 기준**
- `bb_diff_14`: Case 1에서, 가장 가까운 블록버스터 영화와의 거리 변수
- `bb_effect_14`: Case 1에서, 가장 가까운 블록버스터 영화의 영향력에 대한 더미 변수

**Case 2: 28일 기준**
- `bb_diff_28`: Case 2에서, 가장 가까운 블록버스터 영화와의 거리 변수
- `bb_effect_28`: Case 2에서, 가장 가까운 블록버스터 영화의 영향력에 대한 더미 변수

In [15]:
date_array = pd.to_datetime(blockbuster['release_date'], errors='coerce')

df_meta_cleaned_v5['release_date'] = pd.to_datetime(df_meta_cleaned_v5['release_date'], errors='coerce')

NameError: name 'blockbuster' is not defined

## 05-1: Case 1(14일 기준)

### 05-1-1: `bb_diff_14` 변수 생성 함수 정의

In [16]:
# 각 release_date에 대해 가장 가까운 날짜 차이를 계산하고 조건에 맞게 설정
def calculate_min_distance_14(release_date):
    # release_date가 유효한 datetime 값인지 확인
    if pd.isnull(release_date):
        return 0

    # release_date가 date_array보다 큰 경우에만 차이를 계산
    valid_diffs = [(release_date - date).days for date in date_array if release_date > date]

    if valid_diffs:  # 유효한 날짜 차이가 있는 경우
        min_diff = min(valid_diffs)  # 가장 가까운 날짜 차이를 선택
        return 14 - min_diff if min_diff <= 14 else 0  # 14일보다 크면 0으로 설정
    else:
        return 0  # 유효한 날짜 차이가 없으면 0으로 설정

### 05-1-2: `bb_diff_14` 변수 생성

In [17]:
# bb_diff_14 열 생성
df_meta_cleaned_v5['bb_diff_14'] = df_meta_cleaned_v5['release_date'].apply(calculate_min_distance_14)

# 결과 확인
df_meta_cleaned_v5['bb_diff_14'].value_counts()

NameError: name 'date_array' is not defined

### 05-1-3: `bb_effect_14` 변수 생성

In [18]:
# bb_effect_14: 입력값이 14초과이면 1, 이하면 0
df_meta_cleaned_v5['bb_effect_14'] = df_meta_cleaned_v5['bb_diff_14'].apply(lambda x: 0 if int(x) == 0 else 1)

KeyError: 'bb_diff_14'

## 05-2: Case 2(28일 기준)

### 05-2-1: `bb_diff_28` 변수 생성 함수 정의

In [19]:
# 각 release_date에 대해 가장 가까운 날짜 차이를 계산하고 조건에 맞게 설정
def calculate_min_distance_28(release_date):
    # release_date가 유효한 datetime 값인지 확인
    if pd.isnull(release_date):
        return 0

    # release_date가 date_array보다 큰 경우에만 차이를 계산
    valid_diffs = [(release_date - date).days for date in date_array if release_date > date]

    if valid_diffs:  # 유효한 날짜 차이가 있는 경우
        min_diff = min(valid_diffs)  # 가장 가까운 날짜 차이를 선택
        return 28 - min_diff if min_diff <= 28 else 0  # 28일보다 크면 0으로 설정
    else:
        return 0  # 유효한 날짜 차이가 없으면 0으로 설정

### 05-2-2: `bb_diff_28` 변수 생성

In [20]:
# bb_diff_28 열 생성
df_meta_cleaned_v5['bb_diff_28'] = df_meta_cleaned_v5['release_date'].apply(calculate_min_distance_28)

# 결과 확인
df_meta_cleaned_v5['bb_diff_28'].value_counts()

NameError: name 'date_array' is not defined

### 05-2-3: `bb_effect_28` 변수 생성

In [ ]:
# bb_effect_28: 입력값이 28초과이면 1, 이하면 0
df_meta_cleaned_v5['bb_effect_28'] = df_meta_cleaned_v5['bb_diff_28'].apply(lambda x: 0 if int(x) == 0 else 1)

# 06: Export Data

In [21]:
df_meta_cleaned_v5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12631 entries, 0 to 12630
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   budget               12631 non-null  float64
 1   id                   12631 non-null  int64  
 2   popularity           12631 non-null  float64
 3   release_date         12631 non-null  object 
 4   revenue              12631 non-null  float64
 5   runtime              12017 non-null  float64
 6   title                12631 non-null  object 
 7   vote_average         12631 non-null  float64
 8   vote_count           12631 non-null  float64
 9   revenue_api          12502 non-null  float64
 10  budget_api           12497 non-null  float64
 11  cast_1_popularity    10890 non-null  float64
 12  cast_2_popularity    10551 non-null  float64
 13  director_popularity  11856 non-null  float64
dtypes: float64(11), int64(1), object(2)
memory usage: 1.3+ MB


In [158]:
# save point

df_meta_cleaned_v5.to_csv(f'{data_path_write}/fin_df_cleaned_v5.csv', index=False)

## 06-0: REVENUE BUDGET

In [25]:
df_meta_cleaned_v5 = pd.read_csv(f'{data_path_write}/fin_df_cleaned_v5.csv')


매우 큰 값의 budget을 갖고 있는 경우 간혹, budget의 단위가 다르게 기입된 경우가 있는 것으로 보임

- 이 경우 인덱스를 확인해서, api로 끌어온 budget으로 대체

In [27]:
df_meta_cleaned_v5[df_meta_cleaned_v5['budget'] < 500][['budget', 'budget_api']].sort_values(by='budget')

df_meta_cleaned_v5['budget'].value_counts().sort_index()

# 1303, 4881, 10422, 11902, 11308, 4353, 6111, 7642, 3558, 4366, 8923, 3714, 6431

,budget,budget_api
2750,1.0,1.0
8029,1.0,0.0
4029,1.0,1.0
7395,1.0,0.0
7779,1.0,0.0
...,...,...
6313,300.0,300.0
1067,325.0,325.0
8823,400.0,400.0
4326,400.0,400.0


In [175]:
# 기록된 인덱스 리스트
index_list = [1303, 4881, 10422, 11902, 11308, 4353, 6111, 7642, 3558, 4366, 8923, 3714, 6431]

# 인덱스 리스트에 해당하는 행의 budget 값을 budget_api 값으로 대체
for index in index_list:
    if index in df_meta_cleaned_v5.index:  # 인덱스가 데이터프레임에 존재하는지 확인
        df_meta_cleaned_v5.loc[index, 'budget'] = df_meta_cleaned_v5.loc[index, 'budget_api']

# 변경 결과 확인
df_meta_cleaned_v5.loc[index_list, ['budget', 'budget_api']] # 변경된 값 확인

,budget,budget_api
1303,13000000.0,13000000.0
4881,1000000.0,1000000.0
10422,350000.0,350000.0
11902,1500000.0,1500000.0
11308,2000000.0,2000000.0
4353,10000000.0,10000000.0
6111,19000000.0,19000000.0
7642,250000.0,250000.0
3558,15000000.0,15000000.0
4366,15000000.0,15000000.0


In [24]:
df_meta_cleaned_v5[df_meta_cleaned_v5['revenue'] < 500][['revenue', 'revenue_api']]

df_meta_cleaned_v5['revenue'].value_counts().sort_index()

,revenue,revenue_api
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,230522.0
4,0.0,230522.0
...,...,...
12626,0.0,0.0
12627,0.0,0.0
12628,0.0,0.0
12629,0.0,0.0


Target인 revenue가 0인 행이 너무 많기 때문에, 이를 최대한 활용하기 위해 아래와 같은 방식을 취함
1. revenue_api값이 0이거나 결측인 경우, revenue값으로 대체
2. revenue값마저도 0이거나 결측인 경우 최종적으로 행 삭제

In [180]:
# revenue_api 값이 0 또는 결측치이고 revenue 값이 0 또는 결측치인 행을 삭제
df_meta_cleaned_v5 = df_meta_cleaned_v5[
    ~((df_meta_cleaned_v5['revenue_api'] == 0) | (df_meta_cleaned_v5['revenue_api'].isnull())) |
    ~((df_meta_cleaned_v5['revenue'] == 0) | (df_meta_cleaned_v5['revenue'].isnull()))
]

# revenue_api 값이 0 또는 결측치인 경우 revenue 값으로 대체
df_meta_cleaned_v5.loc[
    (df_meta_cleaned_v5['revenue_api'] == 0) | (df_meta_cleaned_v5['revenue_api'].isnull()),
    'revenue_api'
] = df_meta_cleaned_v5.loc[
    (df_meta_cleaned_v5['revenue_api'] == 0) | (df_meta_cleaned_v5['revenue_api'].isnull()),
    'revenue'
]

# 결과 확인
df_meta_cleaned_v5[['revenue', 'revenue_api']].head()

,revenue,revenue_api
3,0.0,230522.0
4,0.0,230522.0
7,115860.0,115860.0
8,115860.0,115860.0
9,0.0,5850000.0


In [182]:
# revenue_api 값으로 revenue 덮어쓰기
df_meta_cleaned_v5['revenue'] = df_meta_cleaned_v5['revenue_api']

# revenue_api 열 삭제
df_meta_cleaned_v5 = df_meta_cleaned_v5.drop(columns=['revenue_api'])

# 결과 확인
df_meta_cleaned_v5.head()

,budget,id,popularity,release_date,revenue,runtime,title,vote_average,vote_count,budget_api,cast_1_popularity,cast_2_popularity,director_popularity,bb_diff_14,bb_effect_14,bb_diff_28,bb_effect_28
3,9.877543e+06,84198,0.501046,2012-03-22,230522.0,84.0,A Place at the Table,6.9,7.0,0.0,3.013,0.001,0.556,0,0,0,0
4,9.877543e+06,84198,0.501046,2012-03-22,230522.0,84.0,A Place at the Table,6.9,7.0,0.0,3.013,0.001,0.556,0,0,0,0
7,3.512454e+06,110428,0.134014,2013-03-13,115860.0,95.0,Camille Claudel 1915,7.0,20.0,3512454.0,0.521,0.001,2.895,0,0,0,0
8,3.512454e+06,110428,0.134014,2013-03-13,115860.0,95.0,Camille Claudel 1915,7.0,20.0,3512454.0,0.521,0.001,2.895,0,0,0,0
9,9.762423e+06,78022,0.645316,2011-09-08,5850000.0,95.0,My Kingdom,4.0,2.0,0.0,5.243,5.514,2.146,0,0,0,0


## 06-0: 마지막 결측치 처리

In [183]:
df_meta_cleaned_v5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3350 entries, 3 to 12621
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   budget               3350 non-null   float64       
 1   id                   3350 non-null   int64         
 2   popularity           3350 non-null   float64       
 3   release_date         3350 non-null   datetime64[ns]
 4   revenue              3350 non-null   float64       
 5   runtime              3302 non-null   float64       
 6   title                3350 non-null   object        
 7   vote_average         3350 non-null   float64       
 8   vote_count           3350 non-null   float64       
 9   budget_api           3345 non-null   float64       
 10  cast_1_popularity    3138 non-null   float64       
 11  cast_2_popularity    3163 non-null   float64       
 12  director_popularity  3211 non-null   float64       
 13  bb_diff_14           3350 non-null   

In [185]:
# 열마다의 왜도, 중앙값, 평균값, 이상치 확인 함수
def analyze_columns(X, columns_to_analyze):
    results = []

    for col in columns_to_analyze:
        skewness = X[col].skew()
        mean_value = X[col].mean()
        median_value = X[col].median()

        # IQR을 사용한 이상치 탐지
        Q1 = X[col].quantile(0.25)
        Q3 = X[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((X[col] < lower_bound) | (X[col] > upper_bound)).sum()  # 이상치의 수

        # 결과 저장
        results.append({
            'Feature': col,
            'Skewness': skewness,
            'Mean': mean_value,
            'Median': median_value,
            'Outliers': outliers
        })

    # 결과를 DataFrame으로 변환
    results_df = pd.DataFrame(results)
    return results_df

# 사용 예시
columns_to_analyze = ['runtime', 'cast_1_popularity', 'cast_2_popularity', 'director_popularity']  # 분석할 컬럼
results_df = analyze_columns(df_meta_cleaned_v5, columns_to_analyze)

# 결과 출력
print(results_df)


               Feature  Skewness        Mean   Median  Outliers
0              runtime  0.917587  106.711084  103.000        81
1    cast_1_popularity  2.587000   22.380237   16.477        89
2    cast_2_popularity  2.935117   18.704696   13.882        89
3  director_popularity  6.101597    5.377179    2.885       314


- runtime을 제외한 popularity들은 skewness가 높다고 판단하여, 중앙값으로 대체
- runtime은 평균값으로

### 최종 결측치 imputation

In [186]:
# runtime 열의 결측치를 평균값으로 대체
df_meta_cleaned_v5['runtime'].fillna(df_meta_cleaned_v5['runtime'].mean(), inplace=True)

# popularity 열들의 결측치를 중앙값으로 대체
popularity_columns = ['cast_1_popularity', 'cast_2_popularity', 'director_popularity']
for column in popularity_columns:
    df_meta_cleaned_v5[column].fillna(df_meta_cleaned_v5[column].median(), inplace=True)

<ipython-input-186-5624a61e47d0>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_meta_cleaned_v5['runtime'].fillna(df_meta_cleaned_v5['runtime'].mean(), inplace=True)
<ipython-input-186-5624a61e47d0>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[c

In [187]:
df_meta_cleaned_v5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3350 entries, 3 to 12621
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   budget               3350 non-null   float64       
 1   id                   3350 non-null   int64         
 2   popularity           3350 non-null   float64       
 3   release_date         3350 non-null   datetime64[ns]
 4   revenue              3350 non-null   float64       
 5   runtime              3350 non-null   float64       
 6   title                3350 non-null   object        
 7   vote_average         3350 non-null   float64       
 8   vote_count           3350 non-null   float64       
 9   budget_api           3345 non-null   float64       
 10  cast_1_popularity    3350 non-null   float64       
 11  cast_2_popularity    3350 non-null   float64       
 12  director_popularity  3350 non-null   float64       
 13  bb_diff_14           3350 non-null   

## 06-1: Case_1_data

In [190]:
df_meta_cleaned_v5.columns

selected_columns_14 = ['budget', 'id', 'popularity', 'release_date', 'revenue', 'runtime',
                       'title', # 해석을 위해 남기는 변수
                       'vote_average', 'vote_count',
                       'cast_1_popularity', 'cast_2_popularity', 'director_popularity',
                        'bb_diff_14', 'bb_effect_14',
                        # 'bb_diff_28', 'bb_effect_28'
                        ]

# Create a new DataFrame with only the selected columns
df_fin_14 = df_meta_cleaned_v5[selected_columns_14]
df_fin_14.info()

df_fin_14.to_csv(f'{data_path_write}/df_fin_14.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 3350 entries, 3 to 12621
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   budget               3350 non-null   float64       
 1   id                   3350 non-null   int64         
 2   popularity           3350 non-null   float64       
 3   release_date         3350 non-null   datetime64[ns]
 4   revenue              3350 non-null   float64       
 5   runtime              3350 non-null   float64       
 6   title                3350 non-null   object        
 7   vote_average         3350 non-null   float64       
 8   vote_count           3350 non-null   float64       
 9   cast_1_popularity    3350 non-null   float64       
 10  cast_2_popularity    3350 non-null   float64       
 11  director_popularity  3350 non-null   float64       
 12  bb_diff_14           3350 non-null   int64         
 13  bb_effect_14         3350 non-null   

## 06-2: Case_2_data

In [191]:
df_meta_cleaned_v5.columns

selected_columns_28 = ['budget', 'id', 'popularity', 'release_date', 'revenue', 'runtime',
                       'title', # 해석을 위해 남기는 변수
                       'vote_average', 'vote_count',
                       'cast_1_popularity', 'cast_2_popularity', 'director_popularity',
                        # 'bb_diff_14', 'bb_effect_14',
                        'bb_diff_28', 'bb_effect_28'
                        ]

# Create a new DataFrame with only the selected columns
df_fin_28 = df_meta_cleaned_v5[selected_columns_28]
df_fin_28.info()

df_fin_28.to_csv(f'{data_path_write}/df_fin_28.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 3350 entries, 3 to 12621
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   budget               3350 non-null   float64       
 1   id                   3350 non-null   int64         
 2   popularity           3350 non-null   float64       
 3   release_date         3350 non-null   datetime64[ns]
 4   revenue              3350 non-null   float64       
 5   runtime              3350 non-null   float64       
 6   title                3350 non-null   object        
 7   vote_average         3350 non-null   float64       
 8   vote_count           3350 non-null   float64       
 9   cast_1_popularity    3350 non-null   float64       
 10  cast_2_popularity    3350 non-null   float64       
 11  director_popularity  3350 non-null   float64       
 12  bb_diff_28           3350 non-null   int64         
 13  bb_effect_28         3350 non-null   